# 21 — Báo cáo tài chính

Mở đầu Track 2. Báo cáo tài chính của FinLens về ở **dạng long** — mỗi dòng là
một (mã, kỳ, chỉ tiêu) — chứ không phải dạng bảng như bản PDF bạn quen nhìn.
Đó là lựa chọn đúng, và notebook này chỉ ra vì sao, rồi dựng lại dạng bảng khi
bạn cần nó.

1. Kỳ nào có số liệu (`periods`)
2. Cây chỉ tiêu: `item_id` · `parent_id` · `level` — và cách in ra cho người đọc
3. **Bốn loại hình doanh nghiệp có bốn cây khác nhau**, và vì sao dạng long xử
   lý được chuyện đó còn dạng bảng thì không
4. Common-size: đọc một báo cáo mà không bị con số tuyệt đối đánh lừa

In [1]:
import sys
from pathlib import Path

GOC = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "finlens_examples").is_dir())
sys.path.insert(0, str(GOC))

import pandas as pd

import finlens
from finlens_examples import ap_dung_theme, duong, heatmap, hom_nay, ty_dong

ap_dung_theme()
client = finlens.client()
HOM_NAY = hom_nay(client)

pd.set_option("display.max_colwidth", 60)

## 1 · Kỳ nào có số liệu

Hỏi trước khi lấy. `periods()` rẻ và nó cho biết chính xác doanh nghiệp đó có
gì — chuỗi bắt đầu từ năm nào, quý nào bị thiếu.

In [2]:
ky = client.financials.periods("HPG")

print(f"HPG có {len(ky)} kỳ, từ {ky['year'].min()} tới {ky['year'].max()}")
print(f"  · năm:  {(ky['quarter'] == 0).sum()} kỳ")
print(f"  · quý:  {(ky['quarter'] > 0).sum()} kỳ")
ky.tail(5)

HPG có 103 kỳ, từ 2004 tới 2026
  · năm:  22 kỳ
  · quý:  81 kỳ


,symbol,company_type,year,quarter,period_label
98,HPG,<NA>,2025,2,Q2 2025
99,HPG,<NA>,2025,3,Q3 2025
100,HPG,<NA>,2025,4,Q4 2025
101,HPG,<NA>,2026,1,Q1 2026
102,HPG,<NA>,2026,2,Q2 2026


`quarter == 0` là **báo cáo năm**, `quarter` từ 1–4 là báo cáo quý. Cột
`period_label` đã ghép sẵn để hiển thị.

## 2 · Một báo cáo, dạng long

Bốn loại báo cáo: `balance_sheet`, `income_statement`, `cash_flow`, và
`cash_flow_direct`.

In [3]:
bs = client.financials.statement(
    "HPG", kind="balance_sheet", period="quarterly", start_year=2026
)

print(f"shape={bs.shape} · đơn vị value = {bs.attrs['finlens']['units']['value']}")
print(f"kỳ có trong frame: {sorted(bs['period_label'].unique())}")
bs.head(4)

shape=(232, 12) · đơn vị value = VND
kỳ có trong frame: ['Q1 2026', 'Q2 2026']


,symbol,company_type,kind,year,quarter,period_label,item_id,line_item,parent_id,level,order_index,value
0,HPG,CT,balance_sheet,2026,2,Q2 2026,1,TÀI SẢN,<NA>,1,0,NaN
1,HPG,CT,balance_sheet,2026,2,Q2 2026,2,A. Tài sản lưu động và đầu tư ngắn hạn,1,2,1,1.199690e+14
2,HPG,CT,balance_sheet,2026,2,Q2 2026,3,I. Tiền và các khoản tương đương tiền,2,3,2,9.782057e+12
3,HPG,CT,balance_sheet,2026,2,Q2 2026,4,1. Tiền,3,4,3,2.221355e+12


Bốn cột dựng nên cây chỉ tiêu:

| Cột | Vai trò |
|---|---|
| `item_id` | định danh chỉ tiêu **trong loại hình doanh nghiệp đó** |
| `parent_id` | chỉ tiêu cha; `<NA>` là gốc |
| `level` | độ sâu, 1 là gốc |
| `order_index` | thứ tự trình bày như bản gốc |

⚠️ `item_id` **không phải mã toàn cục**. `item_id=5` của một ngân hàng và của
một doanh nghiệp phi tài chính là hai chỉ tiêu khác nhau. Luôn đọc kèm
`company_type`.

In [4]:
bs["level"].value_counts().sort_index().rename("số chỉ tiêu").to_frame()

,số chỉ tiêu
level,
1,8
2,8
3,32
4,168
5,16


### In ra như bản gốc

`order_index` giữ đúng thứ tự trình bày, `level` cho biết thụt vào bao nhiêu.
Hai cột đó là đủ — không cần đệ quy trên `parent_id`.

In [5]:
def in_bao_cao(df: pd.DataFrame, ky: str, *, tu_level: int = 4, chia: float = 1e9) -> pd.DataFrame:
    """Dựng lại báo cáo dạng bảng cho một kỳ, thụt lề theo `level`.

    `chia=1e9` đổi VND sang tỷ đồng. `tu_level` cắt bớt các tầng chi tiết —
    một bảng cân đối đầy đủ có 232 dòng, không ai đọc hết.
    """
    mot_ky = df[(df["period_label"] == ky) & (df["level"] <= tu_level)].sort_values("order_index")
    return pd.DataFrame(
        {
            "chỉ tiêu": [
                " " * 4 * (int(lv) - 1) + ten
                for lv, ten in zip(mot_ky["level"], mot_ky["line_item"], strict=True)
            ],
            "tỷ đồng": (mot_ky["value"] / chia).round(0).values,
        }
    ).reset_index(drop=True)


ky_moi_nhat = bs.sort_values(["year", "quarter"])["period_label"].iloc[-1]
in_bao_cao(bs, ky_moi_nhat, tu_level=3).head(20)

,chỉ tiêu,tỷ đồng
0,TÀI SẢN,NaN
1,A. Tài sản lưu động và đầu tư ngắn hạn,119969.0
2,I. Tiền và các khoản tương đương tiền,9782.0
3,II. Các khoản đầu tư tài chính ngắn hạn,27474.0
4,III. Các khoản phải thu ngắn hạn,18040.0
5,IV. Tổng hàng tồn kho,55608.0
6,V. Tài sản ngắn hạn khác,9065.0
7,B. Tài sản cố định và đầu tư dài hạn,158961.0
8,I. Các khoản phải thu dài hạn,2143.0
9,II. Tài sản cố định,132716.0


### Kiểm tra tính nhất quán của cây

Con cộng lại có bằng cha không? Đây là phép kiểm tôi khuyên chạy một lần với
mỗi nguồn dữ liệu mới — nó bắt được lỗi ánh xạ chỉ tiêu mà mắt không thấy.

In [6]:
mot_ky = bs[bs["period_label"] == ky_moi_nhat]

tong_con = mot_ky.dropna(subset=["parent_id"]).groupby("parent_id", observed=True)["value"].sum()
gia_tri_cha = mot_ky.set_index("item_id")["value"]

doi_chieu = pd.DataFrame({"tong_con": tong_con, "gia_tri_cha": gia_tri_cha}).dropna()
# Bỏ các nút cha bằng 0 — chia cho 0 không cho ra một tỷ lệ đọc được
doi_chieu = doi_chieu[doi_chieu["gia_tri_cha"] != 0]
doi_chieu["lech_pct"] = (doi_chieu["tong_con"] / doi_chieu["gia_tri_cha"] - 1) * 100

khop = (doi_chieu["lech_pct"].abs() < 0.01).sum()
print(f"{khop}/{len(doi_chieu)} nút có tổng các con khớp với giá trị cha (sai số < 0,01%)")
print("\nCác nút không khớp (tỷ đồng):")
print(
    doi_chieu[doi_chieu["lech_pct"].abs() >= 0.01]
    .join(mot_ky.set_index("item_id")["line_item"])[
        ["line_item", "tong_con", "gia_tri_cha", "lech_pct"]
    ]
    .assign(
        tong_con=lambda d: (d["tong_con"] / 1e9).round(0),
        gia_tri_cha=lambda d: (d["gia_tri_cha"] / 1e9).round(0),
        lech_pct=lambda d: d["lech_pct"].round(3),
    )
    .to_string()
)

19/21 nút có tổng các con khớp với giá trị cha (sai số < 0,01%)

Các nút không khớp (tỷ đồng):
                               line_item  tong_con  gia_tri_cha  lech_pct
27  B. Tài sản cố định và đầu tư dài hạn  159006.0     158961.0     0.029
57         VI. Tổng tài sản dài hạn khác    7005.0       7050.0    -0.645


Phần lớn khớp tuyệt đối. Vài nút lệch một phần nghìn tới dưới một phần trăm —
đó là những chỗ cây **không cộng dồn thuần tuý**: một khoản mục cha có thể là
tổng của vài con chứ không phải tất cả, hoặc con mang dấu âm (dự phòng, hao
mòn) đã được xử lý sẵn ở giá trị cha.

Kết luận thực hành: **đọc thẳng dòng cha, đừng tự cộng các dòng con.** Và chạy
phép kiểm này một lần với mỗi nguồn dữ liệu mới — nó rẻ, và nó bắt được lỗi ánh
xạ chỉ tiêu mà mắt không thấy.

## 3 · Bốn loại hình, bốn cây chỉ tiêu

`CT` phi tài chính · `NH` ngân hàng · `CK` chứng khoán · `BH` bảo hiểm. Chúng
có cấu trúc báo cáo khác hẳn nhau — ngân hàng không có "hàng tồn kho", công ty
thép không có "tiền gửi của khách hàng".

In [7]:
so_sanh_cay = []
for loai in ["CT", "NH", "CK", "BH"]:
    for bao_cao in ["balance_sheet", "income_statement"]:
        li = client.financials.line_items(com_type=loai, kind=bao_cao)
        so_sanh_cay.append(
            {"loại hình": loai, "báo cáo": bao_cao, "số chỉ tiêu": len(li), "số tầng": li["level"].max()}
        )

pd.DataFrame(so_sanh_cay).pivot(index="loại hình", columns="báo cáo", values="số chỉ tiêu")

báo cáo,balance_sheet,income_statement
loại hình,,
BH,120,62
CK,127,78
CT,116,22
NH,76,23


Nhìn thẳng vào chỗ khác nhau — đầu bảng cân đối của ngân hàng so với doanh
nghiệp phi tài chính:

In [8]:
nh = client.financials.line_items(com_type="NH", kind="balance_sheet")
ct = client.financials.line_items(com_type="CT", kind="balance_sheet")

canh_nhau = pd.DataFrame(
    {
        "CT — phi tài chính": ct.sort_values("order_index")["line_item"].head(10).values,
        "NH — ngân hàng": nh.sort_values("order_index")["line_item"].head(10).values,
    }
)
canh_nhau

,CT — phi tài chính,NH — ngân hàng
0,TÀI SẢN,TÀI SẢN
1,A. Tài sản lưu động và đầu tư ngắn hạn,"I. Tiền mặt, chứng từ có giá trị, ngoại tệ, kim loại quý..."
2,I. Tiền và các khoản tương đương tiền,II. Tiền gửi tại NHNN
3,1. Tiền,III. Tín phiếu kho bạc và các giấy tờ có giá ngắn hạn đủ...
4,2. Các khoản tương đương tiền,"IV. Tiền, vàng gửi tại các TCTD khác và cho vay các TCTD..."
5,II. Các khoản đầu tư tài chính ngắn hạn,"1. Tiền, Vàng gửi tại các TCTD khác"
6,1. Chứng khoán kinh doanh,2. Cho vay các TCTD khác
7,2. Dự phòng giảm giá chứng khoán kinh doanh,3. Dự phòng rủi ro cho vay các TCTD khác
8,3. Đầu tư nắm giữ đến ngày đáo hạn,V. Chứng khoán kinh doanh
9,III. Các khoản phải thu ngắn hạn,1. Chứng khoán kinh doanh


### Và đây là lý do dạng long thắng dạng bảng

Vì mỗi **dòng** mang `company_type` của chính nó, một lời gọi lấy được cả
ngân hàng lẫn doanh nghiệp thép mà không hỏng gì cả. Dạng bảng thì không —
bạn sẽ phải chọn một trong hai cây làm cột.

In [9]:
tron = client.financials.statement(
    ["HPG", "VCB", "SSI", "BVH"],  # CT · NH · CK · BH
    kind="income_statement",
    period="annual",
    start_year=2025,
)

print(f"{len(tron)} dòng, {tron['symbol'].nunique()} mã, {tron['company_type'].nunique()} loại hình")
tron.groupby(["symbol", "company_type"], observed=True).agg(
    so_chi_tieu=("item_id", "nunique"),
    chi_tieu_dau=("line_item", "first"),
)

185 dòng, 4 mã, 4 loại hình


,,so_chi_tieu,chi_tieu_dau
symbol,company_type,,
BVH,BH,62,1- Thu phí bảo hiểm gốc
HPG,CT,22,1. Tổng doanh thu hoạt động kinh doanh
SSI,CK,78,I. DOANH THU HOẠT ĐỘNG
VCB,NH,23,Thu nhập lãi thuần


Bốn mã, bốn cây chỉ tiêu, một frame. Nếu bạn cần dạng bảng thì `pivot` **trong
từng loại hình**, đừng pivot cả frame:

In [10]:
chi_ct = tron[tron["company_type"] == "CT"]
print("Pivot chỉ trong loại hình CT — an toàn:")
print(f"  {chi_ct['symbol'].nunique()} mã · {chi_ct['item_id'].nunique()} chỉ tiêu")

Pivot chỉ trong loại hình CT — an toàn:
  1 mã · 22 chỉ tiêu


## 4 · Common-size: đọc báo cáo mà không bị quy mô đánh lừa

"Chi phí bán hàng 2.000 tỷ" không nói lên điều gì cho tới khi bạn biết doanh
thu là bao nhiêu. Common-size chia mọi dòng của báo cáo kết quả kinh doanh cho
**doanh thu thuần**, và mọi dòng của bảng cân đối cho **tổng tài sản**.

In [11]:
MA = "MWG"
kqkd = client.financials.statement(
    MA, kind="income_statement", period="annual", start_year=2019
)

print("Các chỉ tiêu cấp 1 của báo cáo kết quả kinh doanh:")
print(kqkd[kqkd["level"] == 1][["item_id", "line_item"]].drop_duplicates().head(14).to_string(index=False))

Các chỉ tiêu cấp 1 của báo cáo kết quả kinh doanh:
 item_id                                                            line_item
       1                               1. Tổng doanh thu hoạt động kinh doanh
       2                                      2. Các khoản giảm trừ doanh thu
       3                                           3. Doanh thu thuần (1)-(2)
       4                                                  4. Giá vốn hàng bán
       5                                             5. Lợi nhuận gộp (3)-(4)
       6                                     6. Doanh thu hoạt động tài chính
       7                                                 7. Chi phí tài chính
       9          8. Phần lợi nhuận hoặc lỗ trong công ty liên kết liên doanh
      10                                                  9. Chi phí bán hàng
      11                                     10. Chi phí quản lý doanh nghiệp
      12 11. Lợi nhuận thuần từ hoạt động kinh doanh (5)+(6)-(7)+(8)-(9)-(10)
      13     

In [12]:
# Doanh thu thuần là mẫu số. Tìm nó bằng tên chứ không bằng item_id cứng —
# item_id chỉ ổn định trong một company_type, tên thì đọc được.
DOANH_THU = kqkd[kqkd["line_item"].str.contains("Doanh thu thuần", na=False)]["item_id"].iloc[0]
print(f"item_id của doanh thu thuần (CT): {DOANH_THU}")

mau_so = kqkd[kqkd["item_id"] == DOANH_THU].set_index("period_label")["value"]

common = kqkd[kqkd["level"] <= 2].copy()
common["pct_doanh_thu"] = (
    common["value"] / common["period_label"].map(mau_so) * 100
).round(2)

bang = (
    common.pivot_table(index="line_item", columns="period_label", values="pct_doanh_thu")
    .dropna(how="any")
)
# Giữ các dòng đáng nhìn: bỏ những dòng gần như bằng 0 ở mọi kỳ
bang = bang[bang.abs().max(axis=1) > 0.5]
bang.round(1)

item_id của doanh thu thuần (CT): 3


period_label,2019,2020,2021,2022,2023,2024,2025
line_item,,,,,,,
-Trong đó: Chi phí lãi vay,0.6,0.6,0.6,1.0,1.2,0.8,0.9
1. Tổng doanh thu hoạt động kinh doanh,101.3,101.2,101.0,101.0,100.8,100.7,100.6
10. Chi phí quản lý doanh nghiệp,2.0,3.1,3.1,1.4,1.0,2.6,3.0
11. Lợi nhuận thuần từ hoạt động kinh doanh (5)+(6)-(7)+(8)-(9)-(10),4.9,5.0,5.3,4.9,0.9,3.9,5.6
15. Tổng lợi nhuận kế toán trước thuế (11)+(14),5.0,5.0,5.3,4.5,0.6,3.6,5.5
16. Chi phí thuế TNDN hiện hành,1.2,1.5,1.3,1.3,0.4,1.0,1.1
18. Chi phí thuế TNDN (16)+(17),1.2,1.4,1.3,1.5,0.4,0.8,1.0
19. Lợi nhuận sau thuế thu nhập doanh nghiệp (15)-(18),3.8,3.6,4.0,3.1,0.1,2.8,4.5
2. Các khoản giảm trừ doanh thu,1.3,1.2,1.0,1.0,0.8,0.7,0.6


Đọc theo hàng: biên lợi nhuận gộp co lại hay nở ra, chi phí bán hàng ăn bao
nhiêu phần doanh thu, và tỷ trọng nào đang đổi. Heatmap làm việc đó nhanh hơn:

In [13]:
dang_nhin = bang.loc[bang.index.str.contains("Lợi nhuận gộp|Chi phí bán hàng|Chi phí quản lý|Lợi nhuận sau thuế thu nhập", na=False)]

heatmap(
    dang_nhin,
    tieu_de=f"{MA} — common-size báo cáo kết quả kinh doanh",
    phu_de="Mỗi ô là % doanh thu thuần của năm đó",
    nhan_mau="% doanh thu",
    phan_ky=False,
    dinh_dang_o="%{z:.1f}",
)

⚠️ Thang màu ở đây là **một sắc** (`phan_ky=False`) chứ không phải đỏ↔xanh.
Các con số này đều dương và đo *độ lớn*, không đo *dấu* — dùng thang phân kỳ ở
đây sẽ bịa ra một điểm giữa không tồn tại.

## 5 · So sánh cùng ngành bằng common-size

Đây là chỗ common-size trả công: ba doanh nghiệp bán lẻ khác quy mô hoàn toàn,
nhưng cơ cấu chi phí thì so được.

In [14]:
BAN_LE = ["MWG", "FRT", "DGW"]

nhom = client.financials.statement(
    BAN_LE, kind="income_statement", period="annual", start_year=2025, end_year=2025
)

dt = nhom[nhom["item_id"] == DOANH_THU].set_index("symbol")["value"]
print("Doanh thu thuần 2025 (tỷ đồng):")
print((dt / 1e9).round(0).to_string())

nhom_cs = nhom[nhom["level"] <= 2].assign(
    pct=lambda d: d["value"] / d["symbol"].map(dt) * 100
)

so_sanh = (
    nhom_cs.pivot_table(index="line_item", columns="symbol", values="pct")
    .dropna()
    .round(2)
)
so_sanh = so_sanh[so_sanh.abs().max(axis=1) > 0.5]
so_sanh

Doanh thu thuần 2025 (tỷ đồng):
symbol
DGW     26632.0
FRT     51083.0
MWG    155928.0


symbol,DGW,FRT,MWG
line_item,,,
-Trong đó: Chi phí lãi vay,0.52,0.76,0.94
1. Tổng doanh thu hoạt động kinh doanh,102.30,100.29,100.58
10. Chi phí quản lý doanh nghiệp,0.84,3.28,2.95
11. Lợi nhuận thuần từ hoạt động kinh doanh (5)+(6)-(7)+(8)-(9)-(10),2.51,2.37,5.56
15. Tổng lợi nhuận kế toán trước thuế (11)+(14),2.59,2.39,5.54
16. Chi phí thuế TNDN hiện hành,0.65,0.46,1.10
18. Chi phí thuế TNDN (16)+(17),0.50,0.46,1.00
19. Lợi nhuận sau thuế thu nhập doanh nghiệp (15)-(18),2.08,1.93,4.54
2. Các khoản giảm trừ doanh thu,2.30,0.29,0.58


Quy mô doanh thu chênh nhau nhiều lần, nhưng bảng trên đọc được ngay: ai có
biên gộp dày hơn, ai tiêu nhiều hơn cho bán hàng, và phần chênh lệch đó rơi
xuống lợi nhuận thế nào.

## 6 · Chuỗi thời gian một chỉ tiêu

Việc thường gặp nhất: lấy một dòng của báo cáo qua nhiều năm.

In [15]:
lich_su = client.financials.statement(
    ["HPG", "HSG", "NKG"], kind="income_statement", period="quarterly", start_year=2022
)

LNST = lich_su[lich_su["line_item"].str.contains("Lợi nhuận sau thuế thu nhập doanh nghiệp", na=False)][
    "item_id"
].iloc[0]

lai = (
    lich_su[lich_su["item_id"] == LNST]
    .sort_values(["symbol", "year", "quarter"])
    .assign(
        # pandas 3.0: PeriodIndex(year=, quarter=) đã bỏ — dựng từ chuỗi "2022Q1"
        ky=lambda d: pd.PeriodIndex(
            d["year"].astype(str) + "Q" + d["quarter"].astype(str), freq="Q"
        ).to_timestamp(how="end"),
        lnst_ty=lambda d: d["value"] / 1e9,
    )
)

duong(
    lai,
    x="ky",
    y="lnst_ty",
    theo="symbol",
    tieu_de="Lợi nhuận sau thuế theo quý — nhóm thép",
    phu_de="Cùng chu kỳ ngành, ba biên độ khác nhau",
    nhan_y="tỷ đồng",
    moc_khong=True,
)

## Tổng kết

| Bạn cần | Gọi |
|---|---|
| Kỳ nào có số liệu | `financials.periods("HPG")` |
| Bảng cân đối theo quý | `financials.statement("HPG", kind="balance_sheet", period="quarterly")` |
| Cây chỉ tiêu của một loại hình | `financials.line_items(com_type="NH", kind="balance_sheet")` |
| Nhiều mã khác loại hình | truyền cả danh sách — dạng long xử lý được |

**Bốn điều mang sang notebook sau:**

1. `item_id` chỉ có nghĩa **trong một `company_type`**. Tìm chỉ tiêu bằng tên,
   đừng viết cứng số.
2. Không tự cộng các dòng con để ra dòng cha — một số nút cố ý không cộng dồn.
   Đọc thẳng dòng cha.
3. `order_index` + `level` là đủ để dựng lại bản trình bày gốc.
4. Common-size dùng thang màu **một sắc**, không phải phân kỳ: đây là độ lớn,
   không phải dấu.

---

**Tiếp theo:** [`22_chi_so_va_dupont.ipynb`](22_chi_so_va_dupont.ipynb) — 181
chỉ tiêu tính sẵn, và cái bẫy quý-vs-năm nằm ngay trong cùng một frame.